# Augmented Scene Generation

This notebook creates synthetic UNO scenes from the extracted reference-card crops.
Each scene contains 0 to 6 cards for each player, one center card, and one active-player token.
It saves RGB scenes under `training_images/augmented_scenes` and card segmentation masks under `training_masks/augmented_scenes`.

Masks are generated from per-card visible instances, then each instance is slightly eroded before the final binary mask is written. This creates a small black gap between overlapping cards so a segmenter can learn clean card boundaries.

## Path setup


In [1]:
from pathlib import Path
import sys

candidate_src_dirs = [
    Path.cwd() / "src",
    Path.cwd().parent.parent / "src",
]
for src_dir in candidate_src_dirs:
    if src_dir.is_dir():
        parent = src_dir.parent.resolve()
        if str(parent) not in sys.path:
            sys.path.insert(0, str(parent))
        break
else:
    raise FileNotFoundError("Could not locate do/src directory for imports.")


## Imports


In [ ]:
from src.create_augmented_data import (
    CreateAugmentedDataConfig,
    initialize_create_augmented_data_pipeline,
    plot_card_preview,
    plot_saved_scenes,
    plot_scene_preview,
    run_card_generation,
    run_scene_generation,
    run_scene_preview,
)


## Configuration


In [ ]:
# Most-used quick knobs
SEED = 67
N_AUG_PER_REFERENCE = 500
N_SCENES = 12288
SCENE_SIZE = (1280, 720)
MAX_CARDS_PER_PLAYER = 6
MASK_GAP_PIXELS = 6
PLAYER_CARD_HEIGHT_RANGE = (0.20, 0.22)
CENTER_CARD_HEIGHT_RANGE = (0.20, 0.22)
PLAYER_ROTATION_JITTER_DEG_RANGE = (0.5, 50.0)
AUG_CARD_GENERATION_WORKERS = 0  # 0=auto, or set e.g. 8/12 to cap CPU usage
SCENE_GENERATION_WORKERS = 0  # 0=auto, or set e.g. 8/12 to cap CPU usage

# RGB output compression knobs (masks remain lossless PNG).
CARD_IMAGE_FORMAT = "jpg"
CARD_JPEG_QUALITY = 85
SCENE_IMAGE_FORMAT = "jpg"
SCENE_JPEG_QUALITY = 80

CFG = CreateAugmentedDataConfig(
    seed=SEED,
    n_aug_per_reference=N_AUG_PER_REFERENCE,
    n_scenes=N_SCENES,
    scene_width=SCENE_SIZE[0],
    scene_height=SCENE_SIZE[1],
    max_cards_per_player=MAX_CARDS_PER_PLAYER,
    mask_gap_pixels=MASK_GAP_PIXELS,
    player_card_height_fraction_range=PLAYER_CARD_HEIGHT_RANGE,
    center_card_height_fraction_range=CENTER_CARD_HEIGHT_RANGE,
    player_rotation_jitter_deg_range=PLAYER_ROTATION_JITTER_DEG_RANGE,
    aug_card_generation_workers=AUG_CARD_GENERATION_WORKERS,
    scene_generation_workers=SCENE_GENERATION_WORKERS,
    aug_card_image_format=CARD_IMAGE_FORMAT,
    aug_card_jpeg_quality=CARD_JPEG_QUALITY,
    scene_image_format=SCENE_IMAGE_FORMAT,
    scene_jpeg_quality=SCENE_JPEG_QUALITY,
)
CFG

## Initialize pipeline

Loads reference card crops, prepares output directories, and primes background/token caches.


In [ ]:
state = initialize_create_augmented_data_pipeline(CFG)
state.keys()


## Generate single-card augmentations

Creates `aug_*.jpg` crops for classifier training and the matching `aug.csv` labels file.


In [ ]:
state = run_card_generation(state)
len(state["aug_rows"])


In [ ]:
plot_card_preview(state)


## Preview a synthetic scene

Compose one scene in-memory before kicking off the full generation pass.


In [ ]:
state = run_scene_preview(state)
state["preview_metadata"]["style"], state["preview_metadata"]["active_player"]


In [ ]:
plot_scene_preview(state)


## Generate the augmented scene dataset


In [ ]:
state = run_scene_generation(state)
len(state["scene_metadata"])


## Visual check of saved scenes


In [ ]:
plot_saved_scenes(state)
